# Detección de Fraude de IVA con Neo4j

## Taller Educativo: La Potencia de los Grafos para Detección de Fraude

Autor: mCárdenas @2025

En este notebook verás cómo Neo4j y Cypher permiten detectar patrones complejos de fraude que serían muy difíciles de encontrar con SQL tradicional.

### Objetivos

1. Cargar datos de empresas y transacciones en Neo4j
2. Explorar el grafo con Cypher
3. **Detectar 6 patrones de fraude** usando la potencia de Neo4j
4. Crear un scoring de riesgo combinado


## 1. Configuración y Conexión a Neo4j

In [ ]:
# Importar bibliotecas
from neo4j import GraphDatabase
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Importar consultas predefinidas
import sys
sys.path.append('.')
from consultas_fraude import CONSULTAS_FRAUDE, CONSULTAS_EXPLORACION

print("✓ Bibliotecas importadas")

In [ ]:
# Configuración - AJUSTA TU CONTRASEÑA
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "abc123456"  # ⬅️ CAMBIAR AQUÍ
NEO4J_DATABASE = "fraudedb"

# Conectar
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def ejecutar_consulta(query, parametros=None):
    """Ejecuta una consulta Cypher y retorna resultados"""
    with driver.session(database=NEO4J_DATABASE) as session:
        resultado = session.run(query, parametros or {})
        return [record.data() for record in resultado]

# Verificar conexión
try:
    ejecutar_consulta("RETURN 'Conectado exitosamente' AS mensaje")
    print("✅ Conectado a Neo4j")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Verifica que Neo4j esté corriendo y las credenciales sean correctas")

## 2. Cargar Datos en Neo4j

**Opción 1**: Ejecutar el script de carga:

In [ ]:
# Ejecutar script de carga (asume que ya generaste los datos)
!python cargar_datos_neo4j.py

**Opción 2**: Cargar datos inline (si prefieres hacerlo todo en el notebook):

In [ ]:
# Cargar JSON
with open('datos_fraude_iva.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)

print(f"Datos cargados: {len(datos['empresas'])} empresas, {len(datos['transacciones'])} transacciones")

In [ ]:
# Opcional: Limpiar BD
# ejecutar_consulta("MATCH (n) DETACH DELETE n")

# Crear constraints
constraints = [
    "CREATE CONSTRAINT empresa_nif IF NOT EXISTS FOR (e:Empresa) REQUIRE e.nif IS UNIQUE",
    "CREATE CONSTRAINT directivo_dni IF NOT EXISTS FOR (d:Directivo) REQUIRE d.dni IS UNIQUE"
]

for c in constraints:
    try:
        ejecutar_consulta(c)
    except:
        pass  # Ya existe

print("✓ Constraints creados")

## 3. Verificar Datos en Neo4j

Veamos qué tenemos en la base de datos:

In [ ]:
# Estadísticas generales
query = CONSULTAS_EXPLORACION["estadisticas_generales"]
stats = ejecutar_consulta(query)

print("📊 ESTADÍSTICAS DEL GRAFO\n")
for key, value in stats[0].items():
    print(f"  {key}: {value:,}")

In [ ]:
# Ver algunas empresas
query = """
MATCH (e:Empresa)
RETURN e.nombre AS empresa, e.pais AS país, e.capital_social AS capital, 
       e.empleados AS empleados, e.es_fantasma AS es_fantasma
ORDER BY e.capital_social DESC
LIMIT 10
"""

empresas = ejecutar_consulta(query)
df = pd.DataFrame(empresas)

print("🏢 TOP 10 EMPRESAS POR CAPITAL:\n")
print(df.to_string(index=False))

## 4. LA POTENCIA DE NEO4J: Exploración con Cypher

### 4.1 Empresas Más Conectadas

Con Neo4j es trivial encontrar las empresas más conectadas:

In [ ]:
query = CONSULTAS_EXPLORACION["empresas_mas_conectadas"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("📈 EMPRESAS MÁS CONECTADAS:\n")
print(df.to_string(index=False))

# Visualizar
plt.figure(figsize=(10, 5))
plt.barh(df['empresa'][:10], df['num_conexiones'][:10])
plt.xlabel('Número de Conexiones')
plt.title('Top 10 Empresas por Conectividad')
plt.tight_layout()
plt.show()

### 4.2 Distribución por País

In [ ]:
query = CONSULTAS_EXPLORACION["distribucion_por_pais"]
resultado = ejecutar_consulta(query)

df_pais = pd.DataFrame(resultado)
print("🌍 DISTRIBUCIÓN POR PAÍS:")
print(df_pais.to_string(index=False))



## 5. 🔍 DETECCIÓN DE FRAUDE: Los 6 Patrones

Aquí es donde Neo4j brilla. Vamos a detectar patrones que serían casi imposibles con SQL.



### ⚠️ PATRÓN 1: Cadenas Circulares (Fraude Carrusel)

**El problema**: Detectar empresas que forman ciclos en las transacciones (A→B→C→A).

**En SQL**: Requeriría múltiples JOINs recursivos y sería extremadamente complejo.

**En Neo4j**: Una simple búsqueda de patrones.

In [ ]:
print("🔄 PATRÓN 1: CADENAS CIRCULARES\n")
print(CONSULTAS_FRAUDE["patron_1_cadenas_circulares"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_1_cadenas_circulares"]["query"]
resultado = ejecutar_consulta(query)

if resultado:
    df = pd.DataFrame(resultado)
    print("⚠️  CADENAS CIRCULARES DETECTADAS:")
    for idx, row in df.iterrows():
        print(f"  {idx+1}. {' → '.join(row['empresas_en_cadena'])}")
        print(f"     Longitud: {row['longitud']}, Monto total: €{row['monto_total']:,.0f}")
        print()
else:
    print("No se detectaron cadenas circulares")


### ⚠️ PATRÓN 2: Empresas Fantasma

**El problema**: Empresas creadas recientemente con alto volumen de negocio.

**La ventaja de Neo4j**: Combinar propiedades de nodos con agregaciones de relaciones.

In [ ]:
print("👻 PATRÓN 2: EMPRESAS FANTASMA\n")
print(CONSULTAS_FRAUDE["patron_2_empresas_fantasma"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_2_empresas_fantasma"]["query"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("⚠️  EMPRESAS JÓVENES CON ALTO VOLUMEN:\n")
print(df.to_string(index=False))

# Visualizar
if len(df) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.scatter(df['capital'], df['total_facturado'], c=df['es_fantasma_real'].map({True: 'red', False: 'green'}), alpha=0.6)
    ax1.set_xlabel('Capital Social')
    ax1.set_ylabel('Total Facturado')
    ax1.set_title('Capital vs Facturación')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    df_top = df.nlargest(10, 'total_facturado')
    ax2.barh(df_top['empresa'], df_top['total_facturado'], color=['red' if x else 'green' for x in df_top['es_fantasma_real']])
    ax2.set_xlabel('Total Facturado (€)')
    ax2.set_title('Top 10 por Volumen')
    
    plt.tight_layout()
    plt.show()

### ⚠️ PATRÓN 3: Montos Desproporcionados

**El problema**: Empresas pequeñas con transacciones enormes.

**Neo4j**: Filtra fácilmente por propiedades de nodos Y relaciones simultáneamente.

In [ ]:
print("💰 PATRÓN 3: MONTOS DESPROPORCIONADOS\n")
print(CONSULTAS_FRAUDE["patron_3_montos_desproporcionados"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_3_montos_desproporcionados"]["query"]
resultado = ejecutar_consulta(query)

if resultado:
    df = pd.DataFrame(resultado)
    print("\n⚠️  TRANSACCIONES DESPROPORCIONADAS:\n")
    print(df.to_string(index=False))
else:
    print("No se detectaron montos desproporcionados")

### ⚠️ PATRÓN 4: Directivos que Administran Múltiples Empresas

**El problema**: Encontrar directivos que controlan varias empresas que comercian entre sí.

**Neo4j**: Navega fácilmente por 2 tipos de relaciones (ADMINISTRA y EMITE_FACTURA).

In [ ]:
print("👥 PATRÓN 4: ADMINISTRADORES COMUNES\n")
print(CONSULTAS_FRAUDE["patron_4_administradores_comunes"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_4_administradores_comunes"]["query"]
resultado = ejecutar_consulta(query)

if resultado:
    print("\n⚠️  DIRECTIVOS CON MÚLTIPLES EMPRESAS QUE COMERCIAN ENTRE SÍ:\n")
    for r in resultado:
        print(f"  Directivo: {r['directivo']} (DNI: {r['dni']})")
        print(f"  Administra {r['num_empresas']} empresas: {r['empresas'][:3]}...")
        print(f"  Transacciones internas: {r['num_transacciones_internas']}")
        print(f"  Volumen interno: €{r['total_interno']:,.0f}")
        print()
else:
    print("No se detectaron administradores sospechosos")

### ⚠️ PATRÓN 5: Comunidades Cerradas

**El problema**: Grupos aislados que solo comercian entre ellos.

**Neo4j**: Detectar componentes fuertemente conectados es natural en grafos.

In [ ]:
print("🔒 PATRÓN 5: COMUNIDADES CERRADAS\n")
print(CONSULTAS_FRAUDE["patron_5_comunidades_cerradas"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_5_comunidades_cerradas"]["query"]
resultado = ejecutar_consulta(query)

if resultado:
    print("\n⚠️  COMUNIDADES CERRADAS DETECTADAS:\n")
    for r in resultado:
        print(f"  Grupo de {r['tamaño_grupo']} empresas")
        print(f"  Miembros: {r['grupo_empresas']}")
        print(f"  Trans. internas: {r.get('promedio_trans_internas', 'N/A')}")
        print()
else:
    print("No se detectaron comunidades cerradas")

### ⚠️ PATRÓN 6: Flujos Anómalos Entre Países

**El problema**: Detectar patrones intracomunitarios sospechosos.

**Neo4j**: Agregar por propiedades de nodos relacionados es directo.

In [ ]:
print("🌍 PATRÓN 6: FLUJOS ANÓMALOS ENTRE PAÍSES\n")
print(CONSULTAS_FRAUDE["patron_6_flujos_anomalos"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["patron_6_flujos_anomalos"]["query"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("⚠️  FLUJOS INTRACOMUNITARIOS:\n")
print(df.to_string(index=False))

# Visualizar top flujos
if len(df) > 0:
    plt.figure(figsize=(12, 6))
    rutas = df.apply(lambda x: f"{x['pais_origen']}→{x['pais_destino']}", axis=1)
    plt.barh(rutas[:10], df['total_flujo'][:10])
    plt.xlabel('Volumen Total (€)')
    plt.title('Top 10 Flujos Intracomunitarios')
    plt.tight_layout()
    plt.show()

## 6. 🎯 SCORING DE RIESGO COMBINADO

La verdadera potencia: combinar TODOS los patrones en un score único.

In [ ]:
print("🎯 SCORING DE RIESGO COMBINADO")
print(CONSULTAS_FRAUDE["scoring_riesgo"]["descripcion"])
print("\n" + "="*60 + "\n")

query = CONSULTAS_FRAUDE["scoring_riesgo"]["query"]
resultado = ejecutar_consulta(query)

df_scoring = pd.DataFrame(resultado)
print("🚨 EMPRESAS POR NIVEL DE RIESGO:")
print(df_scoring[['empresa', 'score_riesgo', 'nivel_riesgo', 'num_transacciones', 'volumen_total']].head(20).to_string(index=False))

In [ ]:
# Visualizar distribución de riesgo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribución por nivel
df_scoring['nivel_riesgo'].value_counts().plot(kind='bar', ax=axes[0,0], color=['red', 'orange', 'green'])
axes[0,0].set_title('Distribución por Nivel de Riesgo')
axes[0,0].set_ylabel('Número de Empresas')

# Distribución de scores
axes[0,1].hist(df_scoring['score_riesgo'], bins=20, edgecolor='black')
axes[0,1].set_title('Distribución de Scores')
axes[0,1].set_xlabel('Score de Riesgo')
axes[0,1].axvline(50, color='red', linestyle='--', label='Umbral Alto')
axes[0,1].axvline(30, color='orange', linestyle='--', label='Umbral Medio')
axes[0,1].legend()

# Score vs Volumen
axes[1,0].scatter(df_scoring['volumen_total'], df_scoring['score_riesgo'], 
                   c=df_scoring['nivel_riesgo'].map({'ALTO': 'red', 'MEDIO': 'orange', 'BAJO': 'green'}),
                   alpha=0.6)
axes[1,0].set_xlabel('Volumen Total')
axes[1,0].set_ylabel('Score de Riesgo')
axes[1,0].set_title('Score vs Volumen de Negocio')
axes[1,0].set_xscale('log')
axes[1,0].grid(True, alpha=0.3)

# Top 10 empresas de alto riesgo
top_riesgo = df_scoring.nlargest(10, 'score_riesgo')
axes[1,1].barh(range(len(top_riesgo)), top_riesgo['score_riesgo'], 
                color=[{'ALTO':'red','MEDIO':'orange','BAJO':'green'}[x] for x in top_riesgo['nivel_riesgo']])
axes[1,1].set_yticks(range(len(top_riesgo)))
axes[1,1].set_yticklabels(top_riesgo['empresa'], fontsize=8)
axes[1,1].set_xlabel('Score de Riesgo')
axes[1,1].set_title('Top 10 Empresas de Mayor Riesgo')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. 💡 Conclusiones: La Potencia de Neo4j

### Lo que hicimos

✅ Detectamos **cadenas circulares** con una query simple que en SQL requeriría CTEs recursivos complejos

✅ Encontramos **redes de directivos** navegando 2 tipos de relaciones simultáneamente  

✅ Identificamos **comunidades cerradas** usando la naturaleza de grafos

✅ Combinamos **múltiples patrones** en un scoring unificado

✅ Todo con **consultas declarativas y legibles**

### Por qué Neo4j es superior para esto

1. **Performance**: Navegar relaciones es O(1), no O(n²) como con JOINs
2. **Expresividad**: Los patrones se escriben como se piensan
3. **Flexibilidad**: Añadir nuevas relaciones no requiere reestructurar
4. **Visualización**: El modelo mental coincide con la realidad

### Casos de uso reales

- **Agencias tributarias**: Detección de fraude fiscal
- **Bancos**: Anti-lavado de dinero (AML)
- **Seguros**: Detección de fraude en reclamaciones
- **E-commerce**: Redes de fraude con tarjetas
- **Telecomunicaciones**: Fraude en llamadas



### Próximos Pasos

**Notebook 02**: Usar APOC para análisis más avanzados  
**Notebook 03**: Algoritmos de Graph Data Science (PageRank, Louvain, etc.)



## ¡Perfecto!

Has visto la verdadera potencia de Neo4j para análisis de fraude. Estos patrones serían extremadamente difíciles (o imposibles) de implementar eficientemente en bases de datos relacionales tradicionales.

In [ ]:
# Cerrar conexión
driver.close()
print("✓ Conexión cerrada")